# Fashion-MNIST: Transaction Construction (GPU/CUDA)

Notebook GPU chạy pipeline TC0–TC7 bằng CUDA.jl. Mặc định dùng `configs/tc_smoke_cuda.toml`; có thể chọn `configs/tc_full_cuda.toml` qua biến môi trường `FASHION_TC_CONFIG`. Notebook yêu cầu CUDA hoạt động trước khi nạp dataset.

Mỗi TC có một cell code gọi source dùng chung. Scratch và artifacts trung gian nằm trên ổ local của runtime; checkpoint và TAR được lưu vào output_root trong config.

## 1. Chuẩn bị repository trong Colab

Trên Colab, mount Google Drive tại `/content/gdrive`. Cell kế tiếp chuyển vào `/content/gdrive/MyDrive/[DM]-FIM`; khi chạy local, nó giữ working directory hiện tại.


In [ ]:
colab_drive = "/content/gdrive"
colab_repo = joinpath(colab_drive, "MyDrive", "[DM]-FIM")

if isdir(joinpath(colab_drive, "MyDrive"))
    isfile(joinpath(colab_repo, "Project.toml")) || error(
        "Không tìm thấy Project.toml trong repo Colab: $colab_repo"
    )
    cd(colab_repo)
    println("Colab repository: ", pwd())
else
    println("Không thấy Drive mount tại $colab_drive; giữ working directory hiện tại.")
end


## Chọn cấu hình GPU/CUDA

Mặc định notebook chạy `configs/tc_smoke_cuda.toml`. Để chạy toàn bộ dữ liệu, đặt `FASHION_TC_CONFIG=configs/tc_full_cuda.toml` trước khi khởi động kernel. Notebook chỉ chấp nhận config có `execution.backend = cuda` và dừng sớm nếu thiết bị CUDA không khả dụng.

In [ ]:
const EXPECTED_BACKEND = "cuda"
const CONFIG_OPTIONS = ("configs/tc_smoke_cuda.toml", "configs/tc_full_cuda.toml")
selected_config = get(ENV, "FASHION_TC_CONFIG", first(CONFIG_OPTIONS))
selected_config in CONFIG_OPTIONS || error(
    "Notebook GPU chỉ nhận configs/tc_smoke_cuda.toml hoặc configs/tc_full_cuda.toml; nhận được: $selected_config"
)
ENV["FASHION_TC_CONFIG"] = selected_config
println("Cấu hình GPU đã chọn: ", selected_config)

In [ ]:
using Pkg
using TOML

function find_repo_root(start = pwd())
    current = abspath(start)
    while true
        isfile(joinpath(current, "Project.toml")) && return current
        parent = dirname(current)
        parent == current && error("Không tìm thấy Project.toml từ $start")
        current = parent
    end
end

const REPO_ROOT = find_repo_root()
const CONFIG_RELATIVE_PATH = get(ENV, "FASHION_TC_CONFIG", "configs/tc_smoke_cuda.toml")
const CONFIG_PATH = normpath(joinpath(REPO_ROOT, CONFIG_RELATIVE_PATH))
const CONFIG_PREVIEW = TOML.parsefile(CONFIG_PATH)
const BACKEND = get(CONFIG_PREVIEW["execution"], "backend", "cpu")
BACKEND == EXPECTED_BACKEND || error(
    "Notebook GPU yêu cầu backend=cuda, nhưng config chọn backend=$BACKEND"
)

const GPU_ENV = joinpath(REPO_ROOT, "environments", "fashion_gpu")
Pkg.activate(GPU_ENV)
Pkg.instantiate()
using EclatFIM
@eval using CUDA

CUDA.allowscalar(false)
const TC = EclatFIM.FashionTC
const CONFIG = TC.read_config(CONFIG_PATH)

println("Project: ", Base.active_project())
println("Julia: ", VERSION)
println("Config: ", CONFIG_RELATIVE_PATH)
println("Backend: ", BACKEND)

## 2. Đọc cấu hình đã chọn

Cấu hình được chọn ở cell phía trên. Smoke dùng 100 ảnh train và 20 ảnh validation; full dùng toàn bộ 60.000/10.000 ảnh. Cùng một luồng TC0–TC7 xử lý cả hai chế độ; chỉ TOML thay đổi quy mô, cấu hình K/seed, backend và output.


In [ ]:
println("Config: ", CONFIG_RELATIVE_PATH)
println("Mode: ", CONFIG["run"]["mode"])
println("Backend: ", BACKEND)
println("Output trên Drive: ", normpath(joinpath(REPO_ROOT, CONFIG["run"]["output_root"])))


## Kiểm tra GPU và làm nóng CUDA

Nếu config chọn CUDA, xác minh driver/thiết bị trước khi tải hoặc đọc dataset. Warm-up nhỏ biên dịch kernel trước các phép đo TC4/TC5. GPU không có thì notebook dừng tại đây với thông báo rõ ràng.

In [ ]:
CUDA.allowscalar(false)
gpu = TC.gpu_device_info()
println("Julia: ", VERSION)
println("CUDA.jl: ", gpu.cuda_package_version)
println("GPU: ", gpu.name)
println("VRAM: ", round(gpu.total_memory_bytes / 1024^3; digits=2), " GiB")
if !occursin("T4", gpu.name)
    @warn "Thiết bị hiện tại không phải Tesla T4; số đo tốc độ sẽ không đại diện cho T4 Colab."
end
CUDA.versioninfo()
warmup_result = TC.gpu_warmup!()
display(warmup_result)

## Chuẩn bị dữ liệu và hai thư mục output

Bốn archive IDX `.gz` trên Drive được xác minh rồi chép xuống `/content` để TC0 đọc từ ổ local. Toàn bộ scratch, descriptor và artifacts trung gian ghi vào thư mục làm việc riêng trên `/content`; `output_root` trong config là đích checkpoint/TAR trên Drive.


In [ ]:
using SHA: sha256

# Giữ archive chính thức trên Drive; API kiểm tra checksum trước khi tái sử dụng.
DRIVE_DATA_DIR = joinpath(REPO_ROOT, "data", "raw", "fashion-mnist")
EclatFIM.download_fashion_mnist(DRIVE_DATA_DIR)

# Chép archive đã xác minh xuống ổ local để tránh đọc IDX lặp qua Google Drive.
LOCAL_ROOT = isdir("/content") ? "/content" : tempdir()
DATA_DIR = joinpath(LOCAL_ROOT, "fashion-mnist", "raw")
mkpath(DATA_DIR)

for file in EclatFIM.FASHION_FILES
    source = joinpath(DRIVE_DATA_DIR, file.name)
    local_path = joinpath(DATA_DIR, file.name)
    cp(source, local_path; force=true)
    bytes2hex(open(sha256, local_path)) == file.sha256 || error("Checksum local sai: $(file.name)")
end

OUTPUT_DIR = normpath(joinpath(REPO_ROOT, CONFIG["run"]["output_root"]))
WORK_DIR = joinpath(LOCAL_ROOT, "fashion-mnist", "tc-work", BACKEND,
    CONFIG["run"]["run_id"], string(time_ns()))
mkpath(WORK_DIR)
run_state = TC.prepare_run(CONFIG_PATH; data_dir=DATA_DIR,
    output_root=OUTPUT_DIR, work_root=WORK_DIR)

println("Dữ liệu local: ", DATA_DIR)
println("Scratch local: ", WORK_DIR)
println("Checkpoint/TAR trên Drive: ", OUTPUT_DIR)


## TC0 — Fit ngưỡng từ train

Kiểm tra checksum archive, nạp split gốc và lấy ảnh theo config. Smoke chọn mẫu cân bằng cố định; full dùng toàn bộ train/test. Q10+ luôn được fit trên train của mode đang chọn, sau đó ngưỡng đó dùng chung cho validation.

In [ ]:
tc0_result = TC.run_tc0!(run_state)
display(tc0_result)

## TC1 — Trích patch 4×4

Chia ảnh thành patch theo thứ tự hàng rồi cột. Smoke kiểm tra ghép patch khôi phục đúng ảnh; full dùng cùng kích thước patch và tính toán theo toàn bộ split.

In [ ]:
tc1_result = TC.run_tc1!(run_state)
display(tc1_result)

## TC2 — Lọc patch

Dùng tau của TC0 và alpha trong config để chọn patch. Kết quả gồm số patch giữ/loại theo ảnh và tỉ lệ patch trùng chính xác sau lọc, tính riêng train/validation theo công thức `1 - N_unique / N_total`. Hai patch trùng khi toàn bộ pixel UInt8 giống nhau. CSV `patch_duplicate_summary.csv` lưu số tổng, số unique, số bản trùng và tỉ lệ này.

In [ ]:
tc2_result = TC.run_tc2!(run_state)
display(tc2_result)

## TC3 — Tạo descriptor

Mỗi patch được giữ chuyển thành raw-pixel vector Float32 đã chuẩn hóa. Smoke và full cùng dùng cách biểu diễn, chỉ khác số ảnh/descriptor được xử lý.

In [ ]:
tc3_result = TC.run_tc3!(run_state)
display(tc3_result)

## TC4 — Fit full-batch K-means

Fit chỉ dùng toàn bộ descriptor train sau lọc. CUDA dùng K-means++ và batch 65.536 descriptor để giới hạn bộ nhớ; mỗi vòng vẫn gán toàn bộ dữ liệu rồi mới cập nhật tâm. Buffer khoảng cách được tái sử dụng. `tolerance` là độ đổi objective tuyệt đối giữa hai vòng; chạm `max_iterations` được ghi chưa hội tụ.

Sau từng fit, codebook, metadata và lịch sử iteration được checkpoint lên Drive kèm SHA-256. Báo cáo tách thời gian khởi tạo, fit, assignment, đóng gói TAR và chép lên Drive. Kết thúc fit, toàn bộ train được gán lại theo centroid cuối.


In [ ]:
tc4_result = TC.run_tc4!(run_state)
display(tc4_result)

## TC5 — Gán visual word theo batch

Dùng codebook tương ứng để gán train và validation; CUDA dùng chung kernel khoảng cách với TC4. Word ID hòa khoảng cách chọn ID nhỏ hơn. Assignment trả về CPU giữ nguyên thứ tự ảnh/patch để TC6 tạo transaction. Sau TC5, artifacts mỗi run được đóng TAR và SHA-256 trên Drive.


In [ ]:
tc5_result = TC.run_tc5!(run_state)
display(tc5_result)

## TC6 — Tạo transaction BoVW

Gom word ID theo ảnh, loại trùng và sắp xếp. Mỗi ảnh vẫn có một transaction logic; metadata TID giữ liên kết về split và image ID.

In [ ]:
tc6_result = TC.run_tc6!(run_state)
display(tc6_result)

## TC7 — Kiểm tra và đóng gói artifacts

Smoke reload artifacts từ scratch local, kiểm tra fixture Eclat và support. Full xác minh số transaction/assignment cho từng run. TAR cuối mỗi run được chép lên Drive và chỉ công nhận sau khi SHA-256 bản chép khớp; report và bảng benchmark ghi backend, phiên bản môi trường cùng thời gian từng chặng.


In [ ]:
tc7_result = TC.run_tc7!(run_state; notebook_fresh_kernel=true)
display(tc7_result.summary)

if CONFIG["run"]["mode"] == "smoke"
    println("Report local: ", joinpath(WORK_DIR, "smoke_test_report.json"))
else
    println("Report trên Drive: ", joinpath(OUTPUT_DIR, "full_run_report.toml"))
    println("Bảng benchmark: ", joinpath(OUTPUT_DIR, "benchmark_summary.csv"))
end
println("Checkpoint/TAR trên Drive: ", OUTPUT_DIR)
